# openoppsdb manager

This notebook is connected to `wyattowalsh/openoppsdb`. Schedule it with a daily Kaggle cron cadence such as `0 6 * * *`. Each run installs OpenOpps from GitHub, copies the newest `/kaggle/input/**/openoppsdb.sqlite` snapshot into `/kaggle/working/openoppsdb/openoppsdb.sqlite`, may restore large columns from prior Parquet exports when upgrading legacy thin snapshots, rehydrates the public SQLite snapshot into an operational Alembic schema when needed, runs bounded `openopps jobs sync --metrics-json --freshness-seconds --limit`, captures status and coverage evidence for the private quality gate, then invokes `python -m openopps_kaggle` to backfill derived skill tables, prepare SQLite/CSV/Parquet artifacts, write in-database table and column metadata, prune private evidence, stage the public upload directory, deploy a new dataset version only after the quality gate passes, and attempt best-effort live Kaggle file metadata repair. Local `just kaggle-live-file-metadata` runs use browser-authenticated cookies for the authoritative Kaggle DataBundle checklist repair.


In [ ]:
#@title Initialize
from __future__ import annotations

import csv
import hashlib
import json
import os
from pathlib import Path
import shutil
import sqlite3
import subprocess
import sys
import time
from datetime import UTC, datetime
import urllib.request

DATASET_ID = os.environ.get(
    "OPENOPPS_KAGGLE_DATASET",
    "wyattowalsh/openoppsdb",
)
PACKAGE_SPEC = os.environ.get(
    "OPENOPPS_PACKAGE_SPEC",
    "git+https://github.com/wyattowalsh/openopps.git@main",
)
OUTPUT_DIR = Path(
    os.environ.get(
        "OPENOPPS_KAGGLE_OUTPUT_DIR",
        "/kaggle/working/openoppsdb",
    )
)
DB_PATH = OUTPUT_DIR / "openoppsdb.sqlite"
PUBLIC_UPLOAD_DIR = OUTPUT_DIR / "public-upload"
RUNTIME_PACKAGE_DIR = OUTPUT_DIR / "openopps_kaggle"
RUNTIME_MANIFEST_PATH = OUTPUT_DIR / "runtime-manifest.json"
GENERATOR_SCRIPT = RUNTIME_PACKAGE_DIR
RUNTIME_PACKAGE_URL = os.environ.get(
    "OPENOPPS_RUNTIME_PACKAGE_URL",
    "file:///kaggle/input/openoppsdb-manager-runtime/openopps_kaggle/",
)
GENERATOR_SCRIPT_URL = RUNTIME_PACKAGE_URL
RUNTIME_PACKAGE_SHA256 = os.environ.get(
    "OPENOPPS_RUNTIME_PACKAGE_SHA256",
    "10c7648a1748c435d19de1459db8d90dbaef22fb5cf231c980274e3a60e6f0db",
).strip().lower()
GENERATOR_SCRIPT_SHA256 = RUNTIME_PACKAGE_SHA256
GENERATOR_SCRIPT_VERIFIED_SHA256 = None
RUNTIME_PACKAGE_VERIFIED_SHA256 = None
KAGGLE_CREDENTIAL_ENV_NAMES = {
    "KAGGLE_API_TOKEN",
    "KAGGLE_API_V1_TOKEN_PATH",
    "KAGGLE_CONFIG_DIR",
    "KAGGLE_IAP_TOKEN",
    "KAGGLE_KEY",
    "KAGGLE_URL_BASE",
    "KAGGLE_USERNAME",
    "KAGGLE_USER_SECRETS_TOKEN",
}
KAGGLE_INPUT_DIR = Path("/kaggle/input")
INPUT_DB_GLOB = "**/openoppsdb.sqlite"
INPUT_SOURCES_PARQUET_GLOB = "**/exports/parquet/sources.parquet"
INPUT_BOARDS_PARQUET_GLOB = "**/exports/parquet/boards.parquet"
INPUT_JOB_VERSIONS_PARQUET_GLOB = "**/exports/parquet/job_versions.parquet"
INPUT_JOB_PAYLOAD_SNAPSHOTS_PARQUET_GLOB = "**/exports/parquet/job_payload_snapshots.parquet"
OPENOPPS_SYNC_ENV_DEFAULTS = {
    "OPENOPPS_BOARD_CONCURRENCY": "80",
    "OPENOPPS_HTTP_TIMEOUT": "20",
    "OPENOPPS_JOB_ROUTE_FRESHNESS_SECONDS": "86400",
    "OPENOPPS_JOB_ROUTE_TIMEOUT_SECONDS": "180",
    "OPENOPPS_MAX_CONNECTIONS": "120",
    "OPENOPPS_PROVIDER_CONCURRENCY": "80",
    "OPENOPPS_RETRY_ATTEMPTS": "2",
    "OPENOPPS_SOURCE_CONCURRENCY": "40",
    "OPENOPPS_SOURCE_FRESHNESS_SECONDS": "86400",
    "OPENOPPS_SOURCE_TIMEOUT_SECONDS": "120"
}
PUBLIC_METADATA_TABLES = {"openopps_tables", "openopps_columns"}
APP_TABLE_NAMES = ('sources',
 'boards',
 'board_providers',
 'jobs',
 'job_versions',
 'job_version_locations',
 'job_version_skills',
 'job_version_skill_keywords',
 'job_version_bullets',
 'job_payload_snapshots',
 'job_sync_runs',
 'job_sync_observations')
APP_PRIMARY_KEY_COLUMNS = {'board_providers': ('id',),
 'boards': ('key',),
 'job_payload_snapshots': ('id',),
 'job_sync_observations': ('id',),
 'job_sync_runs': ('id',),
 'job_version_bullets': ('id',),
 'job_version_locations': ('id',),
 'job_version_skill_keywords': ('id',),
 'job_version_skills': ('id',),
 'job_versions': ('id',),
 'jobs': ('id',),
 'sources': ('key',)}
PARQUET_RESTORE_TABLES = ('job_version_locations', 'job_version_skills', 'job_version_skill_keywords', 'job_version_bullets')
PUBLIC_SNAPSHOT_JSON_DEFAULTS = {
    ("sources", "version"): "{}",
    ("sources", "raw_metadata"): "{}",
    ("sources", "extra_payload"): "{}",
    ("boards", "source_keys"): "[]",
    ("boards", "source_board_keys"): "{}",
    ("boards", "markets"): "[]",
    ("boards", "locations"): "[]",
    ("boards", "raw_payload"): "{}",
    ("boards", "extra_payload"): "{}",
    ("board_providers", "raw_payload"): "{}",
    ("board_providers", "extra_payload"): "{}",
    ("jobs", "extra_payload"): "{}",
    ("job_versions", "version"): "{}",
    ("job_versions", "locations"): "[]",
    ("job_versions", "compensation"): "{}",
    ("job_versions", "responsibilities"): "[]",
    ("job_versions", "qualifications"): "[]",
    ("job_versions", "skills"): "[]",
    ("job_versions", "job_description"): "{}",
    ("job_versions", "extra_payload"): "{}",
    ("job_payload_snapshots", "payload"): "{}",
}
KAGGLE_SYNC_TIMEOUT_SECONDS = float(
    os.environ.get(
        "OPENOPPS_KAGGLE_SYNC_TIMEOUT_SECONDS",
        "3300",
    )
)
KAGGLE_JOB_ROUTE_LIMIT = int(
    os.environ.get(
        "OPENOPPS_KAGGLE_JOB_ROUTE_LIMIT",
        "120",
    )
)
KAGGLE_METADATA_WAIT_SECONDS = float(
    os.environ.get("OPENOPPS_KAGGLE_METADATA_WAIT_SECONDS", "120")
)
KAGGLE_METADATA_POLL_SECONDS = float(
    os.environ.get("OPENOPPS_KAGGLE_METADATA_POLL_SECONDS", "30")
)
KAGGLE_CREDENTIALS_ERROR = (
    "Kaggle API credentials are required to publish openoppsdb. "
    "Configure KAGGLE_USERNAME and KAGGLE_KEY as Kaggle notebook secrets "
    "before running the manager."
)
KAGGLE_SECRET_RETRIES = int(os.environ.get("OPENOPPS_KAGGLE_SECRET_RETRIES", "30"))
KAGGLE_SECRET_RETRY_SECONDS = float(
    os.environ.get("OPENOPPS_KAGGLE_SECRET_RETRY_SECONDS", "10")
)
KAGGLE_SECRET_URL_BASE = os.environ.get(
    "OPENOPPS_KAGGLE_SECRET_URL_BASE",
    "https://www.kaggle.com",
)
KAGGLE_SECRET_LOOKUP_ERRORS: dict[str, str] = {}
KAGGLE_SECRET_SERVICE_DIAGNOSTIC_EMITTED = False

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def describe_secret_exception(exc: BaseException) -> str:
    parts = [type(exc).__name__]
    message = str(exc).strip()
    if message:
        parts.append(message[:240])
    cause = getattr(exc, "__cause__", None)
    if cause is not None:
        cause_parts = [type(cause).__name__]
        cause_message = str(cause).strip()
        if cause_message:
            cause_parts.append(cause_message[:240])
        reason = getattr(cause, "reason", None)
        if reason is not None:
            reason_message = str(reason).strip()
            if reason_message:
                cause_parts.append(
                    f"reason={type(reason).__name__}:{reason_message[:180]}"
                )
        parts.append("cause=" + " | ".join(cause_parts))
    return " | ".join(parts)


def emit_kaggle_secret_service_diagnostics() -> None:
    global KAGGLE_SECRET_SERVICE_DIAGNOSTIC_EMITTED
    if KAGGLE_SECRET_SERVICE_DIAGNOSTIC_EMITTED:
        return
    KAGGLE_SECRET_SERVICE_DIAGNOSTIC_EMITTED = True
    print(
        "Kaggle secret-service diagnostics:",
        json.dumps(
            {
                "KAGGLE_IAP_TOKEN_present": bool(os.environ.get("KAGGLE_IAP_TOKEN")),
                "KAGGLE_SECRET_URL_BASE": KAGGLE_SECRET_URL_BASE,
                "KAGGLE_URL_BASE_runtime": os.environ.get("KAGGLE_URL_BASE"),
                "KAGGLE_USER_SECRETS_TOKEN_present": bool(
                    os.environ.get("KAGGLE_USER_SECRETS_TOKEN")
                ),
            },
            sort_keys=True,
        ),
    )
    try:
        with urllib.request.urlopen(KAGGLE_SECRET_URL_BASE, timeout=10) as response:
            print(f"Kaggle URL reachability check succeeded: status={response.status}")
    except Exception as exc:
        print(
            "Kaggle URL reachability check failed: "
            f"{describe_secret_exception(exc)}"
        )


def normalize_kaggle_notebook_secret(value: object) -> str | None:
    if isinstance(value, str) and value.strip():
        return value.strip()
    return None


def read_kaggle_notebook_secrets() -> tuple[str | None, str | None]:
    last_key = None
    last_username = None
    runtime_url_base = os.environ.get("KAGGLE_URL_BASE")
    os.environ["KAGGLE_URL_BASE"] = KAGGLE_SECRET_URL_BASE
    try:
        for attempt in range(1, KAGGLE_SECRET_RETRIES + 1):
            key_error = None
            username_error = None
            try:
                from kaggle_secrets import UserSecretsClient
            except Exception as exc:
                KAGGLE_SECRET_LOOKUP_ERRORS["kaggle_secrets"] = type(exc).__name__
                print(
                    f"Kaggle notebook secrets client unavailable: {type(exc).__name__}"
                )
                return last_key, last_username

            user_secrets = UserSecretsClient()
            try:
                secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
            except Exception as exc:
                key_error = type(exc).__name__
                KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_KEY"] = key_error
                if attempt == 1:
                    emit_kaggle_secret_service_diagnostics()
                print(
                    f"KAGGLE_KEY notebook secret lookup failed "
                    f"(attempt {attempt}/{KAGGLE_SECRET_RETRIES}): "
                    f"{key_error}"
                )
                if attempt in {1, KAGGLE_SECRET_RETRIES}:
                    print(f"KAGGLE_KEY lookup detail: {describe_secret_exception(exc)}")
                secret_value_0 = None

            try:
                secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")
            except Exception as exc:
                username_error = type(exc).__name__
                KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_USERNAME"] = username_error
                print(
                    f"KAGGLE_USERNAME notebook secret lookup failed "
                    f"(attempt {attempt}/{KAGGLE_SECRET_RETRIES}): "
                    f"{username_error}"
                )
                if attempt in {1, KAGGLE_SECRET_RETRIES}:
                    print(
                        "KAGGLE_USERNAME lookup detail: "
                        f"{describe_secret_exception(exc)}"
                    )
                secret_value_1 = None

            key = normalize_kaggle_notebook_secret(secret_value_0)
            username = normalize_kaggle_notebook_secret(secret_value_1)
            if key:
                last_key = key
                KAGGLE_SECRET_LOOKUP_ERRORS.pop("KAGGLE_KEY", None)
            elif key_error is None:
                KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_KEY"] = "NotFound"
                print(
                    f"KAGGLE_KEY not found in Kaggle notebook secrets "
                    f"(attempt {attempt}/{KAGGLE_SECRET_RETRIES})."
                )

            if username:
                last_username = username
                KAGGLE_SECRET_LOOKUP_ERRORS.pop("KAGGLE_USERNAME", None)
            elif username_error is None:
                KAGGLE_SECRET_LOOKUP_ERRORS["KAGGLE_USERNAME"] = "NotFound"
                print(
                    f"KAGGLE_USERNAME not found in Kaggle notebook secrets "
                    f"(attempt {attempt}/{KAGGLE_SECRET_RETRIES})."
                )

            if key and username:
                return key, username

            if attempt < KAGGLE_SECRET_RETRIES:
                time.sleep(KAGGLE_SECRET_RETRY_SECONDS)
                continue
        return last_key, last_username
    finally:
        if runtime_url_base is None:
            os.environ.pop("KAGGLE_URL_BASE", None)
        else:
            os.environ["KAGGLE_URL_BASE"] = runtime_url_base


def load_kaggle_notebook_secrets() -> None:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        print("KAGGLE_USERNAME and KAGGLE_KEY already present in environment.")
        return

    key, username = read_kaggle_notebook_secrets()

    if os.environ.get("KAGGLE_USERNAME"):
        print("KAGGLE_USERNAME already present in environment.")
    elif username:
        os.environ["KAGGLE_USERNAME"] = username
        print("KAGGLE_USERNAME loaded from Kaggle notebook secrets.")

    if os.environ.get("KAGGLE_KEY"):
        print("KAGGLE_KEY already present in environment.")
    elif key:
        os.environ["KAGGLE_KEY"] = key
        print("KAGGLE_KEY loaded from Kaggle notebook secrets.")


def has_kaggle_credentials() -> bool:
    load_kaggle_notebook_secrets()
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    token_path = os.environ.get("KAGGLE_API_V1_TOKEN_PATH")
    return bool(
        os.environ.get("KAGGLE_API_TOKEN")
        or (token_path and Path(token_path).expanduser().exists())
        or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
        or kaggle_json.exists()
    )


def require_kaggle_credentials() -> None:
    if not has_kaggle_credentials():
        if KAGGLE_SECRET_LOOKUP_ERRORS:
            details = ", ".join(
                f"{key}={value}"
                for key, value in sorted(KAGGLE_SECRET_LOOKUP_ERRORS.items())
            )
            raise RuntimeError(f"{KAGGLE_CREDENTIALS_ERROR} Lookup diagnostics: {details}")
        raise RuntimeError(KAGGLE_CREDENTIALS_ERROR)


def run(
    command: list[str],
    *,
    env: dict[str, str] | None = None,
    timeout_seconds: float | None = None,
) -> None:
    print("+", " ".join(command))
    subprocess.run(command, check=True, env=env, timeout=timeout_seconds)


def run_json(
    command: list[str],
    output_path: Path,
    *,
    env: dict[str, str] | None = None,
    timeout_seconds: float | None = None,
) -> dict:
    print("+", " ".join(command), ">", output_path)
    try:
        completed = subprocess.run(
            command,
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout_seconds,
        )
    except subprocess.TimeoutExpired as exc:
        if exc.stdout:
            print(exc.stdout)
        if exc.stderr:
            print(exc.stderr, file=sys.stderr)
        timeout_label = (
            f"{timeout_seconds:g}" if timeout_seconds is not None else "unknown"
        )
        raise TimeoutError(
            f"Command exceeded {timeout_label}s: {' '.join(command)}"
        ) from exc
    if completed.returncode:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr, file=sys.stderr)
        completed.check_returncode()
    data = json.loads(completed.stdout)
    output_path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n")
    print(f"Wrote {output_path}")
    return data


def kaggle_dataset_status() -> dict:
    completed = subprocess.run(
        ["kaggle", "datasets", "status", DATASET_ID, "--format", "json"],
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return json.loads(completed.stdout)


def install_openopps() -> None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", PACKAGE_SPEC, "kaggle"])


def required_runtime_package_sha256() -> str:
    if not GENERATOR_SCRIPT_SHA256:
        raise RuntimeError(
            "OPENOPPS_RUNTIME_PACKAGE_SHA256 is required before downloading "
            "the OpenOpps Kaggle runtime package."
        )
    if len(GENERATOR_SCRIPT_SHA256) != 64 or any(
        character not in "0123456789abcdef" for character in GENERATOR_SCRIPT_SHA256
    ):
        raise RuntimeError(
            "OPENOPPS_RUNTIME_PACKAGE_SHA256 must be a lowercase SHA-256 hex digest."
        )
    return GENERATOR_SCRIPT_SHA256


def verify_runtime_package_manifest() -> str:
    global GENERATOR_SCRIPT_VERIFIED_SHA256, RUNTIME_PACKAGE_VERIFIED_SHA256
    expected_digest = required_runtime_package_sha256()
    manifest = json.loads(RUNTIME_MANIFEST_PATH.read_text(encoding="utf-8"))
    actual_digest = str(manifest.get("sha256") or "")
    if actual_digest != expected_digest:
        raise RuntimeError(
            "OpenOpps Kaggle runtime package checksum mismatch: "
            f"expected={expected_digest} actual={actual_digest}"
        )
    files = manifest.get("files")
    if not isinstance(files, dict):
        raise RuntimeError("runtime-manifest.json missing files map")
    for rel, digest in sorted(files.items()):
        if rel.startswith("openopps_kaggle/"):
            path = RUNTIME_PACKAGE_DIR / rel.removeprefix("openopps_kaggle/")
        elif rel == "runtime-manifest.json":
            path = RUNTIME_MANIFEST_PATH
        else:
            path = OUTPUT_DIR / rel
        if not path.is_file():
            raise FileNotFoundError(f"Missing runtime package file: {path}")
        file_digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if file_digest != digest:
            raise RuntimeError(
                f"Runtime package file checksum mismatch for {rel}: "
                f"expected={digest} actual={file_digest}"
            )
    GENERATOR_SCRIPT_VERIFIED_SHA256 = actual_digest
    RUNTIME_PACKAGE_VERIFIED_SHA256 = actual_digest
    return actual_digest


def runtime_probe_env() -> dict[str, str]:
    env = os.environ.copy()
    for key in list(env):
        if key.startswith("KAGGLE_") or key in KAGGLE_CREDENTIAL_ENV_NAMES:
            env.pop(key, None)
    return env


generator_probe_env = runtime_probe_env


def download_runtime_package() -> None:
    required_runtime_package_sha256()
    runtime_input = next(
        (path for path in KAGGLE_INPUT_DIR.glob("**/openoppsdb-manager-runtime") if path.is_dir()),
        KAGGLE_INPUT_DIR / "openoppsdb-manager-runtime",
    )
    manifest_source = runtime_input / "runtime-manifest.json"
    package_source = runtime_input / "openopps_kaggle"
    if not manifest_source.is_file() or not package_source.is_dir():
        raise RuntimeError(
            "Runtime package must be attached via openoppsdb-manager-runtime input; "
            f"expected {manifest_source} and {package_source}"
        )
    shutil.copy2(manifest_source, RUNTIME_MANIFEST_PATH)
    if RUNTIME_PACKAGE_DIR.exists():
        shutil.rmtree(RUNTIME_PACKAGE_DIR)
    shutil.copytree(package_source, RUNTIME_PACKAGE_DIR)
    digest = verify_runtime_package_manifest()
    completed = subprocess.run(
        [sys.executable, "-m", "openopps_kaggle", "--help"],
        env={**runtime_probe_env(), "PYTHONPATH": str(OUTPUT_DIR)},
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    if completed.returncode:
        completed.check_returncode()
    help_text = completed.stdout + completed.stderr
    required_flags = [
        "--skip-notebooks",
        "--stage-public-upload-dir",
        "--quality-report",
    ]
    missing_flags = [flag for flag in required_flags if flag not in help_text]
    if missing_flags:
        raise RuntimeError(
            "Downloaded OpenOpps Kaggle runtime package is incompatible; "
            f"missing CLI flags: {', '.join(missing_flags)}"
        )
    print(
        "OpenOpps Kaggle runtime package ready:",
        json.dumps(
            {
                "manifest": str(RUNTIME_MANIFEST_PATH),
                "package": str(RUNTIME_PACKAGE_DIR),
                "sha256": digest,
            },
            sort_keys=True,
        ),
    )


def run_openopps_kaggle(
    args: list[str],
    *,
    timeout_seconds: float | None = None,
) -> None:
    if not RUNTIME_PACKAGE_DIR.exists():
        download_runtime_package()
    else:
        verify_runtime_package_manifest()
    env = os.environ.copy()
    env["PYTHONPATH"] = str(RUNTIME_PACKAGE_DIR.parent)
    run(
        [sys.executable, "-m", "openopps_kaggle", *args],
        env=env,
        timeout_seconds=timeout_seconds,
    )


def try_run_openopps_kaggle(args: list[str]) -> bool:
    try:
        run_openopps_kaggle(args)
    except Exception as exc:
        print(
            "Kaggle live metadata repair failed after successful dataset publish; "
            "continuing so the scheduled snapshot run completes. "
            "Run `just kaggle-live-file-metadata` from a browser-authenticated local "
            f"environment to retry databundle repair. Error: {type(exc).__name__}: {exc}"
        )
        return False
    return True


def emit_disk_usage(label: str, path: Path = OUTPUT_DIR) -> None:
    usage_path = path if path.exists() else path.parent
    usage = shutil.disk_usage(usage_path)
    print(
        "OpenOpps disk usage:",
        json.dumps(
            {
                "freeBytes": usage.free,
                "label": label,
                "path": str(usage_path),
                "totalBytes": usage.total,
                "usedBytes": usage.used,
            },
            sort_keys=True,
        ),
    )


def run_sync_metrics(
    output_path: Path,
    *,
    env: dict[str, str],
    timeout_seconds: float | None,
) -> dict:
    return run_json(
        [
            "openopps",
            "jobs",
            "sync",
            "--metrics-json",
            "--freshness-seconds",
            env.get("OPENOPPS_JOB_ROUTE_FRESHNESS_SECONDS", "86400"),
            "--limit",
            str(KAGGLE_JOB_ROUTE_LIMIT),
        ],
        output_path,
        env=env,
        timeout_seconds=timeout_seconds,
    )


def sqlite_sidecars(path: Path) -> tuple[Path, ...]:
    return tuple(path.with_name(path.name + suffix) for suffix in ("-wal", "-shm", "-journal"))


def remove_sqlite_sidecars(path: Path) -> None:
    for sidecar in sqlite_sidecars(path):
        sidecar.unlink(missing_ok=True)


def sqlite_table_names(db_path: Path) -> set[str]:
    if not db_path.exists():
        return set()
    with sqlite3.connect(db_path) as conn:
        return {
            row[0]
            for row in conn.execute(
                "SELECT name FROM sqlite_master WHERE type = 'table'"
            )
        }


def is_public_snapshot_db(db_path: Path) -> bool:
    table_names = sqlite_table_names(db_path)
    return (
        "alembic_version" not in table_names
        and set(APP_TABLE_NAMES).issubset(table_names)
        and PUBLIC_METADATA_TABLES.issubset(table_names)
    )


def sanitize_public_snapshot_json_columns(snapshot_path: Path) -> dict[str, int]:
    sanitized: dict[str, int] = {}
    with sqlite3.connect(snapshot_path) as conn:
        for (table_name, column_name), default_json in PUBLIC_SNAPSHOT_JSON_DEFAULTS.items():
            table_sql = quote_identifier(table_name)
            column_sql = quote_identifier(column_name)
            table_exists = conn.execute(
                "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = ?",
                (table_name,),
            ).fetchone()
            if not table_exists:
                continue
            column_exists = any(
                str(row[1]) == column_name
                for row in conn.execute(f"PRAGMA table_info({table_sql})")
            )
            if not column_exists:
                continue
            bad_count = int(
                conn.execute(
                    f"SELECT count(*) FROM {table_sql} "
                    f"WHERE {column_sql} IS NOT NULL "
                    f"AND json_valid({column_sql}) = 0"
                ).fetchone()[0]
            )
            if not bad_count:
                continue
            conn.execute(
                f"UPDATE {table_sql} SET {column_sql} = ? "
                f"WHERE {column_sql} IS NOT NULL "
                f"AND json_valid({column_sql}) = 0",
                (default_json,),
            )
            sanitized[f"{table_name}.{column_name}"] = bad_count
        conn.commit()
    if sanitized:
        print(
            "Sanitized invalid public snapshot JSON before rehydrate:",
            json.dumps(sanitized, sort_keys=True),
        )
    return sanitized


def copy_latest_input_db() -> None:
    db_candidates = sorted(KAGGLE_INPUT_DIR.glob(INPUT_DB_GLOB))
    if db_candidates:
        source_db = max(db_candidates, key=lambda path: path.stat().st_mtime)
        shutil.copy2(source_db, DB_PATH)
        print(f"Copied prior OpenOpps DB snapshot from {source_db} to {DB_PATH}")
        restore_projected_sqlite_columns_from_input_exports()
    else:
        print("No prior OpenOpps DB snapshot found; creating a new ledger.")


def quote_identifier(value: str) -> str:
    return '"' + value.replace('"', '""') + '"'


def quote_string_literal(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def qualified_table(schema: str, table_name: str) -> str:
    if schema not in {"main", "public_snapshot"}:
        raise ValueError(f"Unsupported SQLite schema: {schema}")
    return f"{schema}.{quote_identifier(table_name)}"


def table_columns(conn: sqlite3.Connection, schema: str, table_name: str) -> list[str]:
    if schema not in {"main", "public_snapshot"}:
        raise ValueError(f"Unsupported SQLite schema: {schema}")
    return [
        row[1]
        for row in conn.execute(
            f"PRAGMA {schema}.table_info({quote_string_literal(table_name)})"
        )
    ]


def public_snapshot_table_count(
    conn: sqlite3.Connection,
    schema: str,
    table_name: str,
) -> int:
    return int(
        conn.execute(
            f"SELECT count(*) FROM {qualified_table(schema, table_name)}"
        ).fetchone()[0]
    )


def validate_public_snapshot_table(
    conn: sqlite3.Connection,
    table_name: str,
) -> list[str]:
    source_columns = table_columns(conn, "public_snapshot", table_name)
    target_columns = table_columns(conn, "main", table_name)
    if not source_columns:
        raise RuntimeError(f"Public OpenOpps snapshot is missing table: {table_name}")
    missing_columns = [column for column in target_columns if column not in source_columns]
    if missing_columns:
        raise RuntimeError(
            "Public OpenOpps snapshot table "
            f"{table_name} is missing required columns: {', '.join(missing_columns)}"
        )

    primary_key_columns = APP_PRIMARY_KEY_COLUMNS[table_name]
    missing_key_columns = [
        column for column in primary_key_columns if column not in source_columns
    ]
    if missing_key_columns:
        raise RuntimeError(
            "Public OpenOpps snapshot table "
            f"{table_name} is missing primary key columns: "
            f"{', '.join(missing_key_columns)}"
        )
    key_expressions = ", ".join(quote_identifier(column) for column in primary_key_columns)
    null_predicate = " OR ".join(
        f"{quote_identifier(column)} IS NULL" for column in primary_key_columns
    )
    null_key_count = int(
        conn.execute(
            f"SELECT count(*) FROM {qualified_table('public_snapshot', table_name)} "
            f"WHERE {null_predicate}"
        ).fetchone()[0]
    )
    if null_key_count:
        raise RuntimeError(
            "Public OpenOpps snapshot table "
            f"{table_name} has {null_key_count} rows with null primary keys."
        )
    duplicate_key_count = int(
        conn.execute(
            "SELECT count(*) FROM ("
            f"SELECT {key_expressions}, count(*) AS duplicate_count "
            f"FROM {qualified_table('public_snapshot', table_name)} "
            f"GROUP BY {key_expressions} HAVING duplicate_count > 1"
            ")"
        ).fetchone()[0]
    )
    if duplicate_key_count:
        raise RuntimeError(
            "Public OpenOpps snapshot table "
            f"{table_name} has {duplicate_key_count} duplicate primary keys."
        )
    return target_columns


def import_public_snapshot_tables(snapshot_path: Path) -> dict[str, int]:
    summary: dict[str, int] = {}
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            f"ATTACH DATABASE {quote_string_literal(snapshot_path.as_posix())} "
            "AS public_snapshot"
        )
        try:
            for table_name in APP_TABLE_NAMES:
                target_columns = validate_public_snapshot_table(conn, table_name)
                source_count = public_snapshot_table_count(
                    conn, "public_snapshot", table_name
                )
                existing_count = public_snapshot_table_count(
                    conn, "main", table_name
                )
                if existing_count:
                    raise RuntimeError(
                        "Cannot import public OpenOpps snapshot into non-empty "
                        f"table {table_name}; found {existing_count} existing rows."
                    )
                column_sql = ", ".join(
                    quote_identifier(column) for column in target_columns
                )
                conn.execute(
                    f"INSERT INTO {qualified_table('main', table_name)} ({column_sql}) "
                    f"SELECT {column_sql} "
                    f"FROM {qualified_table('public_snapshot', table_name)}"
                )
                imported_count = public_snapshot_table_count(
                    conn, "main", table_name
                )
                if imported_count != source_count:
                    raise RuntimeError(
                        "Public OpenOpps snapshot import count mismatch for "
                        f"{table_name}: source={source_count}, imported={imported_count}."
                    )
                summary[table_name] = imported_count
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            conn.execute("DETACH DATABASE public_snapshot")
    return summary


def restore_derived_public_snapshot_tables_from_parquet() -> dict[str, int]:
    restored: dict[str, int] = {}
    if not DB_PATH.exists():
        return restored
    import polars as pl

    for table_name in PARQUET_RESTORE_TABLES:
        parquet_candidates = sorted(
            KAGGLE_INPUT_DIR.glob(f"**/exports/parquet/{table_name}.parquet")
        )
        if not parquet_candidates:
            continue
        source_parquet = max(parquet_candidates, key=lambda path: path.stat().st_mtime)
        with sqlite3.connect(DB_PATH) as conn:
            table_exists = conn.execute(
                "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = ?",
                (table_name,),
            ).fetchone()
            if not table_exists:
                continue
            existing_count = public_snapshot_table_count(conn, "main", table_name)
            if existing_count:
                continue
            target_columns = table_columns(conn, "main", table_name)
            if not target_columns:
                continue

            frame = pl.read_parquet(source_parquet, columns=target_columns)
            if frame.height == 0:
                continue
            column_sql = ", ".join(quote_identifier(column) for column in target_columns)
            placeholders = ", ".join("?" for _ in target_columns)
            insert_sql = (
                f"INSERT INTO {qualified_table('main', table_name)} ({column_sql}) "
                f"VALUES ({placeholders})"
            )
            batch: list[tuple] = []
            inserted = 0
            for row in frame.iter_rows(named=False):
                batch.append(row)
                if len(batch) >= 10_000:
                    conn.executemany(insert_sql, batch)
                    inserted += len(batch)
                    batch.clear()
            if batch:
                conn.executemany(insert_sql, batch)
                inserted += len(batch)
            conn.commit()
        restored[table_name] = inserted
        print(
            "Restored public snapshot table from Parquet:",
            json.dumps(
                {
                    "source": str(source_parquet),
                    "table": table_name,
                    "rows": inserted,
                },
                sort_keys=True,
            ),
        )
    return restored


def rehydrate_public_snapshot_for_openopps(env: dict[str, str]) -> bool:
    if not DB_PATH.exists() or not is_public_snapshot_db(DB_PATH):
        return False
    snapshot_path = DB_PATH.with_name("openoppsdb-public-snapshot.sqlite")
    snapshot_path.unlink(missing_ok=True)
    remove_sqlite_sidecars(snapshot_path)
    print(
        "Rehydrating public OpenOppsDB snapshot into operational SQLite schema."
    )
    shutil.move(DB_PATH, snapshot_path)
    remove_sqlite_sidecars(DB_PATH)
    run(["openopps", "admin", "db", "init"], env=env)
    sanitized_json = sanitize_public_snapshot_json_columns(snapshot_path)
    summary = import_public_snapshot_tables(snapshot_path)
    parquet_restored = restore_derived_public_snapshot_tables_from_parquet()
    print(
        "Rehydrated public OpenOppsDB snapshot:",
        json.dumps(
            {
                "source": str(snapshot_path),
                "tables": summary,
                "parquetRestoredTables": parquet_restored,
                "sanitizedJson": sanitized_json,
            },
            sort_keys=True,
        ),
    )
    return True


def restore_projected_sqlite_table_columns(
    *,
    parquet_glob: str,
    table_name: str,
    key_column: str,
    column_names: list[str],
    restore_invalid_json: bool = False,
) -> None:
    parquet_candidates = sorted(KAGGLE_INPUT_DIR.glob(parquet_glob))
    if not parquet_candidates or not DB_PATH.exists():
        return
    source_parquet = max(parquet_candidates, key=lambda path: path.stat().st_mtime)
    table_sql = quote_identifier(table_name)
    restore_predicates = [
        f"{quote_identifier(column)} IS NULL" for column in column_names
    ]
    if restore_invalid_json:
        restore_predicates.extend(
            f"({quote_identifier(column)} IS NOT NULL "
            f"AND json_valid({quote_identifier(column)}) = 0)"
            for column in column_names
        )
    restore_condition = " OR ".join(restore_predicates)
    with sqlite3.connect(DB_PATH) as conn:
        restore_candidate_count = int(
            conn.execute(
                f"SELECT count(*) FROM {table_sql} WHERE {restore_condition}"
            ).fetchone()[0]
        )
    if restore_candidate_count == 0:
        return

    import polars as pl

    restore_csv = OUTPUT_DIR / f"_restore_{table_name}.csv"
    restore_columns = [key_column, *column_names]
    pl.scan_parquet(source_parquet).select(restore_columns).sink_csv(restore_csv)
    restored_rows = 0
    csv.field_size_limit(sys.maxsize)
    with sqlite3.connect(DB_PATH) as conn, restore_csv.open(
        newline="", encoding="utf-8"
    ) as handle:
        reader = csv.DictReader(handle)
        restore_table = f"restore_{table_name}"
        restore_table_sql = quote_identifier(restore_table)
        column_defs = ", ".join(
            f"{quote_identifier(column)} TEXT" for column in restore_columns
        )
        key_column_sql = quote_identifier(key_column)
        conn.execute(
            f"CREATE TEMP TABLE {restore_table_sql} "
            f"({column_defs}, PRIMARY KEY ({key_column_sql}))"
        )
        batch = []
        for row in reader:
            batch.append(tuple(row[column] for column in restore_columns))
            if len(batch) >= 1000:
                conn.executemany(
                    f"INSERT OR REPLACE INTO {restore_table_sql} VALUES "
                    f"({', '.join('?' for _ in restore_columns)})",
                    batch,
                )
                restored_rows += len(batch)
                batch.clear()
        if batch:
            conn.executemany(
                f"INSERT OR REPLACE INTO {restore_table_sql} VALUES "
                f"({', '.join('?' for _ in restore_columns)})",
                batch,
            )
            restored_rows += len(batch)
        assignments = ", ".join(
            f"{quote_identifier(column)} = ("
            f"SELECT {restore_table_sql}.{quote_identifier(column)} "
            f"FROM {restore_table_sql} "
            f"WHERE {restore_table_sql}.{key_column_sql} = {table_sql}.{key_column_sql})"
            for column in column_names
        )
        conn.execute(
            f"UPDATE {table_sql} SET {assignments} "
            f"WHERE {restore_condition} AND EXISTS ("
            f"SELECT 1 FROM {restore_table_sql} "
            f"WHERE {restore_table_sql}.{key_column_sql} = {table_sql}.{key_column_sql})"
        )
        conn.commit()
    restore_csv.unlink(missing_ok=True)
    print(
        "Restored projected SQLite values from prior Parquet export:",
        json.dumps(
            {
                "source": str(source_parquet),
                "table": table_name,
                "columns": column_names,
                "missingBefore": restore_candidate_count,
                "restoreRows": restored_rows,
                "restoreInvalidJson": restore_invalid_json,
            },
            sort_keys=True,
        ),
    )


def restore_projected_sqlite_columns_from_input_exports() -> None:
    restore_projected_sqlite_table_columns(
        parquet_glob=INPUT_SOURCES_PARQUET_GLOB,
        table_name="sources",
        key_column="key",
        column_names=["raw_metadata"],
        restore_invalid_json=True,
    )
    restore_projected_sqlite_table_columns(
        parquet_glob=INPUT_BOARDS_PARQUET_GLOB,
        table_name="boards",
        key_column="key",
        column_names=[
            "source_board_keys",
            "markets",
            "locations",
        ],
        restore_invalid_json=True,
    )
    restore_projected_sqlite_table_columns(
        parquet_glob=INPUT_BOARDS_PARQUET_GLOB,
        table_name="boards",
        key_column="key",
        column_names=["raw_payload"],
    )
    restore_projected_sqlite_table_columns(
        parquet_glob=INPUT_JOB_VERSIONS_PARQUET_GLOB,
        table_name="job_versions",
        key_column="id",
        column_names=[
            "description",
            "description_html",
            "job_description",
            "responsibilities",
            "qualifications",
            "skills",
            "compensation",
        ],
    )
    restore_projected_sqlite_table_columns(
        parquet_glob=INPUT_JOB_VERSIONS_PARQUET_GLOB,
        table_name="job_versions",
        key_column="id",
        column_names=["locations"],
        restore_invalid_json=True,
    )
    restore_projected_sqlite_table_columns(
        parquet_glob=INPUT_JOB_PAYLOAD_SNAPSHOTS_PARQUET_GLOB,
        table_name="job_payload_snapshots",
        key_column="id",
        column_names=["payload"],
    )


require_kaggle_credentials()
install_openopps()
download_runtime_package()
copy_latest_input_db()


In [ ]:
openopps_env = os.environ.copy()
openopps_env["OPENOPPS_DB_URL"] = f"sqlite:///{DB_PATH}"
openopps_env["OPENOPPS_CACHE_ENABLED"] = "false"
for key, value in OPENOPPS_SYNC_ENV_DEFAULTS.items():
    openopps_env.setdefault(key, value)

rehydrate_public_snapshot_for_openopps(openopps_env)
run(["openopps", "admin", "db", "init"], env=openopps_env)
print(f"OpenOpps bounded jobs sync timeout: {KAGGLE_SYNC_TIMEOUT_SECONDS:g}s")
print(f"OpenOpps bounded jobs sync route limit: {KAGGLE_JOB_ROUTE_LIMIT}")
sync_metrics = run_sync_metrics(
    OUTPUT_DIR / "sync_metrics.json",
    env=openopps_env,
    timeout_seconds=KAGGLE_SYNC_TIMEOUT_SECONDS,
)
status = run_json(
    ["openopps", "status", "--json"],
    OUTPUT_DIR / "status.json",
    env=openopps_env,
)
coverage = run_json(
    ["openopps", "providers", "coverage", "--json"],
    OUTPUT_DIR / "coverage.json",
    env=openopps_env,
)
emit_disk_usage("after_sync")


In [ ]:
emit_disk_usage("before_artifact_export")
generator_args = [
    "--output-dir",
    str(OUTPUT_DIR),
    "--data-db",
    str(DB_PATH),
    "--mutate-data-db-for-upload",
    "--sync-metrics",
    str(OUTPUT_DIR / "sync_metrics.json"),
    "--status-json",
    str(OUTPUT_DIR / "status.json"),
    "--coverage-json",
    str(OUTPUT_DIR / "coverage.json"),
    "--quality-report",
    str(OUTPUT_DIR / "snapshot-quality.json"),
    "--prune-private-upload-files",
    "--stage-public-upload-dir",
    str(PUBLIC_UPLOAD_DIR),
    "--skip-notebooks",
]
empty_snapshot_explanation = os.environ.get("OPENOPPS_EMPTY_SNAPSHOT_EXPLANATION")
if empty_snapshot_explanation:
    generator_args.extend(["--empty-snapshot-explanation", empty_snapshot_explanation])
run_openopps_kaggle(generator_args)
emit_disk_usage("after_artifact_export")

for path in sorted(PUBLIC_UPLOAD_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PUBLIC_UPLOAD_DIR), path.stat().st_size)


In [ ]:
message = f"Scheduled OpenOpps active-job snapshot {datetime.now(UTC).isoformat()}"
require_kaggle_credentials()
previous_status = kaggle_dataset_status()
previous_version = int(previous_status.get("current_version_number") or 0)
print(
    "OpenOpps live dataset version before publish:",
    json.dumps(previous_status, sort_keys=True),
)

emit_disk_usage("before_dataset_publish")
run([
    "kaggle",
    "datasets",
    "version",
    "-p",
    str(PUBLIC_UPLOAD_DIR),
    "-m",
    message,
    "-q",
    "-t",
    "-r",
    "zip",
])
expected_version = previous_version + 1
metadata_repair_ok = try_run_openopps_kaggle([
    "--output-dir",
    str(OUTPUT_DIR),
    "--skip-notebooks",
    "--wait-live-dataset-ready",
    "--wait-live-dataset-min-version",
    str(expected_version),
    "--wait-live-dataset-timeout-seconds",
    str(KAGGLE_METADATA_WAIT_SECONDS),
    "--wait-live-dataset-poll-seconds",
    str(KAGGLE_METADATA_POLL_SECONDS),
    "--update-live-file-metadata",
])
print(
    "OpenOpps live metadata repair:",
    json.dumps(
        {
            "databundle": "skipped_in_notebook",
            "mode": "kaggle-api",
            "ok": metadata_repair_ok,
        },
        sort_keys=True,
    ),
)
run(["kaggle", "datasets", "status", DATASET_ID, "--format", "json"])
run(["kaggle", "datasets", "files", DATASET_ID, "--page-size", "200"])
